
# 02. Reducción dimensional



## Objetivo

Estudiar las cinco estrategias utilizadas para reducir las características
candidatas:

1. Importancia mediante Random Forest.
2. RFE.
3. Selección de lags.
4. PCA.
5. Método combinado.

La reducción dimensional genera diferentes datasets que posteriormente
serán utilizados para entrenar las MLP.


In [1]:

from pathlib import Path
from datetime import datetime

RUTA_PROYECTO = Path(
    r"C:\Users\marco\Documentos\investigacion"
    r"\machine_learning_idalina\6_redes_neuronales"
)

RUTA_DATOS_RAW = RUTA_PROYECTO / "2_datos" / "1_raw"

RUTA_PROCESADOS = RUTA_PROYECTO / "2_datos" / "2_procesados"

RUTA_RESULTADOS = RUTA_PROCESADOS / "resultados_mlp"

RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Proyecto:")
print(RUTA_PROYECTO)

print("\nDatos originales:")
print(RUTA_DATOS_RAW)

print("\nDatos procesados:")
print(RUTA_PROCESADOS)

print("\nResultados MLP:")
print(RUTA_RESULTADOS)


Proyecto:
C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales

Datos originales:
C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\1_raw

Datos procesados:
C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados

Resultados MLP:
C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\2_datos\2_procesados\resultados_mlp


[Video de apoyo a la gestión de carpetas para reducción dimensional]()

## 1. Bibliotecas necesarias para diferentes formas de Reducción dimensional

In [ ]:

import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA


# ¿Quién es RobustScaler?  



# RobustScaler (sklearn.preprocessing)

`RobustScaler` es un transformador de sklearn que escala características numéricas usando **estadísticos robustos a outliers**, en lugar de la media y la desviación estándar (como hace `StandardScaler`) o el mínimo/máximo (como `MinMaxScaler`).

## Cómo funciona RobustScaler

Para cada característica, la transformación es:

```
X_escalado = (X - mediana) / IQR
```

Donde:
- **Mediana**: el valor central de la distribución
- **IQR (rango intercuartílico)**: por defecto, `Q3 - Q1` (percentil 75 - percentil 25)

Al usar mediana e IQR en vez de media y desviación estándar, los valores atípicos (outliers) tienen mucho menos influencia sobre el resultado del escalado.



## Uso básico



In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Parámetros principales

| Parámetro | Default | Descripción |
|---|---|---|
| `with_centering` | `True` | Si `False`, no resta la mediana (útil para datos dispersos/sparse) |
| `with_scaling` | `True` | Si `False`, no divide por el IQR |
| `quantile_range` | `(25.0, 75.0)` | Puedes cambiar el rango de percentiles usados como "IQR" |
| `unit_variance` | `False` | Si `True`, ajusta el escalado para que, bajo una distribución normal, la varianza resultante sea 1 |



## Atributos tras el ajuste

- `center_`: la mediana de cada característica (si `with_centering=True`)
- `scale_`: el IQR de cada característica (si `with_scaling=True`)

## ¿Cuándo usarlo?

- Cuando tus datos tienen **outliers significativos** que distorsionarían la media/desviación estándar.
- Cuando quieres una alternativa robusta a `StandardScaler` sin eliminar los outliers del dataset.
- No garantiza que los valores queden en un rango fijo (a diferencia de `MinMaxScaler`); los outliers pueden seguir teniendo valores escalados grandes, pero no dominan el cálculo del escalado.



## Ejemplo comparativo



In [7]:
import numpy as np
from sklearn.preprocessing import RobustScaler, StandardScaler

X = np.array([[1], [2], [3], [4], [100]])  # 100 es un outlier

print(StandardScaler().fit_transform(X).ravel())
# La media y std se ven muy afectadas por el 100

print(RobustScaler().fit_transform(X).ravel())
# La mediana e IQR casi no cambian por el outlier


[-0.53828462 -0.51265202 -0.48701942 -0.46138681  1.99934286]
[-1.  -0.5  0.   0.5 48.5]


Con `StandardScaler`, el outlier "comprime" el resto de los valores hacia un rango pequeño porque infla la desviación estándar. Con `RobustScaler`, los primeros cuatro valores mantienen una escala más razonable entre sí.



# Ejemplo con datos reales o cómo se compara con `MinMaxScaler` y `StandardScaler` en un pipeline 

En este ejemplo se comparan los tres escaladores con datos que incluyen outliers, y luego un ejemplo de uso dentro de un pipeline.

## 1. Comparación visual de los tres escaladores



In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler

# Datos con un outlier evidente
np.random.seed(42)
datos_normales = np.random.normal(loc=50, scale=5, size=20)
datos_con_outlier = np.append(datos_normales, [500])  # outlier extremo

X = datos_con_outlier.reshape(-1, 1)

resultados = pd.DataFrame({
    "original": X.ravel(),
    "StandardScaler": StandardScaler().fit_transform(X).ravel(),
    "MinMaxScaler": MinMaxScaler().fit_transform(X).ravel(),
    "RobustScaler": RobustScaler().fit_transform(X).ravel(),
})

print(resultados.describe().round(1))


       original  StandardScaler  MinMaxScaler  RobustScaler
count      21.0            21.0          21.0          21.0
mean       70.6             0.0           0.1           3.9
std        98.5             1.0           0.2          17.8
min        40.4            -0.3           0.0          -1.5
25%        47.2            -0.2           0.0          -0.3
50%        48.8            -0.2           0.0           0.0
75%        52.7            -0.2           0.0           0.7
max       500.0             4.5           1.0          81.7


**Lo que verás:**
- `StandardScaler`: la desviación estándar se infla por el outlier, así que los 20 valores "normales" quedan comprimidos en un rango muy pequeño (casi todos cerca de 0).
- `MinMaxScaler`: aún peor para este caso, porque el rango completo lo define el outlier (0 a 500), aplastando el resto de los datos casi todos cerca de 0.
- `RobustScaler`: los 20 valores normales mantienen una dispersión razonable y reconocible; el outlier simplemente queda con un valor escalado grande, pero no distorsiona a los demás.

## 2. Uso dentro de un Pipeline (caso típico)



In [4]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

# Supongamos que X, y son tus datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("modelo", Ridge())
])

pipeline.fit(X_train, y_train)
predicciones = pipeline.predict(X_test)


NameError: name 'y' is not defined

Usar un `Pipeline` es importante aquí: así el escalador se ajusta (`fit`) **solo** con los datos de entrenamiento, y luego se aplica (`transform`) a los datos de prueba sin "filtrar" información — evita el data leakage.



## 3. Ajustando el rango de percentiles

Si el IQR estándar (25–75) te parece demasiado estrecho o amplio, puedes ajustarlo:

```python
scaler = RobustScaler(quantile_range=(10.0, 90.0))
X_train_scaled = scaler.fit_transform(X_train)
```



Esto usa el percentil 10 y 90 en lugar de 25/75, dándote un rango de referencia más amplio (menos sensible aún a valores extremos cerca de los cuartiles, aunque sigue ignorando el 10% en cada cola).



## ¿Cuándo elegir cada uno?

| Situación | Recomendación |
|---|---|
| Datos limpios, sin outliers relevantes | `StandardScaler` |
| Necesitas rango fijo (ej. 0-1) y no hay outliers | `MinMaxScaler` |
| Hay outliers que no quieres eliminar pero tampoco que dominen el escalado | `RobustScaler` |
| Modelos basados en distancias (KNN, SVM, redes neuronales) con datos ruidosos | `RobustScaler` suele ser buena opción |
| Modelos basados en árboles (Random Forest, XGBoost) | Generalmente no necesitan escalado |


## 2. Volvamos al proceso de reducción dimensional a la etapa de: Cargar datos

In [6]:
from pathlib import Path
RUTA_PROYECTO = Path(
    r"C:\Users\marco\Documentos\investigacion"
    r"\machine_learning_idalina\6_redes_neuronales"
)

RUTA_DATOS_RAW = RUTA_PROYECTO / "2_datos" / "1_raw"

RUTA_EXCEL = (
    RUTA_DATOS_RAW /
    "2_meteo_epi_2021-2026_1_rezagos.xlsx"
)

df = pd.read_excel(
    RUTA_EXCEL,
    parse_dates=["fecha"]
)

df = df.sort_values("fecha").reset_index(drop=True)


COLUMNAS_NO_FEATURE = [
    "fecha",
    "casos_dengue"
]

features = [
    c for c in df.columns
    if c not in COLUMNAS_NO_FEATURE
]

print("Features candidatas:", len(features))


Features candidatas: 170


## 3. Selección mediante importancia de Random Forest

In [6]:

def seleccionar_por_importancia_random_forest(
    df,
    features,
    threshold=0.005
):

    X = df[features].values
    y = df["casos_dengue"].values

    imputer = SimpleImputer(strategy="median")

    X = imputer.fit_transform(X)

    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )

    rf.fit(X, y)

    importancias = pd.DataFrame({
        "feature": features,
        "importance": rf.feature_importances_
    })

    importancias = (
        importancias
        .sort_values(
            "importance",
            ascending=False
        )
    )

    seleccionadas = (
        importancias[
            importancias["importance"] > threshold
        ]["feature"]
        .tolist()
    )

    return seleccionadas, importancias


## 4. RFE

In [7]:

def seleccionar_rfe(
    df,
    features,
    n_features=20
):

    X = df[features].values
    y = df["casos_dengue"].values

    imputer = SimpleImputer(
        strategy="median"
    )

    X = imputer.fit_transform(X)

    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )

    selector = RFE(
        rf,
        n_features_to_select=n_features,
        step=1
    )

    selector.fit(X, y)

    seleccionadas = [
        features[i]
        for i in range(len(features))
        if selector.support_[i]
    ]

    return seleccionadas


## 5. Lags óptimos

In [9]:

def seleccionar_lags_optimos(df):

    lags = [
        f"casos_dengue_lag_{i}"
        for i in range(1, 13)
        if f"casos_dengue_lag_{i}" in df.columns
    ]

    correlaciones = []

    X = df[lags].values
    y = df["casos_dengue"].values

    X = SimpleImputer(
        strategy="median"
    ).fit_transform(X)

    for i, col in enumerate(lags):

        corr = np.corrcoef(
            X[:, i],
            y
        )[0, 1]

        if not np.isnan(corr):

            correlaciones.append(
                (col, abs(corr))
            )

    correlaciones.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        col
        for col, corr in correlaciones
        if corr > 0.3
    ]


## 6. PCA

In [10]:

def aplicar_pca(
    df,
    features,
    n_components=10
):

    X = df[features].values

    X = SimpleImputer(
        strategy="median"
    ).fit_transform(X)

    scaler = RobustScaler()

    X_scaled = scaler.fit_transform(X)

    pca = PCA(
        n_components=n_components
    )

    X_pca = pca.fit_transform(X_scaled)

    columnas = [
        f"PC_{i+1}"
        for i in range(n_components)
    ]

    df_pca = pd.DataFrame(
        X_pca,
        columns=columnas
    )

    df_pca["fecha"] = df["fecha"].values
    df_pca["casos_dengue"] = (
        df["casos_dengue"].values
    )

    return (
        df_pca,
        columnas,
        pca
    )


## 7. Comparar las estrategias

In [11]:

features_imp, importancias = (
    seleccionar_por_importancia_random_forest(
        df,
        features
    )
)

features_rfe = seleccionar_rfe(
    df,
    features,
    n_features=20
)

features_lags = seleccionar_lags_optimos(df)

df_pca, features_pca, pca = aplicar_pca(
    df,
    features,
    n_components=10
)

comparacion = pd.DataFrame({
    "Método": [
        "Importancia",
        "RFE",
        "Lags",
        "PCA"
    ],

    "Features": [
        len(features_imp),
        len(features_rfe),
        len(features_lags),
        len(features_pca)
    ]
})

comparacion


,Método,Features
0,Importancia,9
1,RFE,20
2,Lags,12
3,PCA,10



### Observación metodológica

PCA no selecciona variables originales. Produce componentes principales.

Por tanto:

- Importancia → selección de features.
- RFE → selección de features.
- Lags → selección de features.
- PCA → transformación/reducción de dimensionalidad.
